# Backend API Test Notebook
Use this notebook to smoke-test FastAPI endpoints.

Set `API_BASE` in the next cell to either your Render URL or `http://localhost:3000` for local dev.

> **Note:** The Render free tier spins down after 15 minutes of inactivity. The first request after a cold start can take 30–60 seconds — just wait and retry.


## 0) Start

In [1]:
import json
from typing import Any
import requests

LOCAL_URL = "http://localhost:3000"
RENDER_URL = "https://london-explorer.onrender.com"  # ← paste your Render URL here
TIMEOUT_SECONDS = 30

# Auto-select: use local if it's up, otherwise fall back to Render
try:
    requests.get(f"{LOCAL_URL}/health", timeout=2)
    API_BASE = LOCAL_URL
    print(f"✓ Local server detected — using {API_BASE}")
except requests.exceptions.ConnectionError:
    API_BASE = RENDER_URL
    print(f"✓ Local server not running — using {API_BASE}")


✓ Local server detected — using http://localhost:3000


In [2]:


def call_api(path: str, params: dict[str, Any] | None = None) -> Any:
    url = f"{API_BASE}{path}"
    response = requests.get(url, params=params, timeout=TIMEOUT_SECONDS)
    try:
        response.raise_for_status()
    except requests.HTTPError as exc:
        detail = response.text
        raise requests.HTTPError(f"{exc}\nResponse body: {detail}") from exc
    return response.json()

def preview(payload: Any, max_items: int = 3):
    if isinstance(payload, dict) and "data" in payload and isinstance(payload["data"], list):
        data = payload["data"]
        summary = {k: v for k, v in payload.items() if k != "data"}
        print("Summary:")
        print(json.dumps(summary, indent=2))
        print("\nData preview:")
        print(json.dumps(data[:max_items], indent=2))
        print(f"\nData length: {len(data)}")
        return

    print(json.dumps(payload, indent=2))


## 1) Health Check

In [3]:
health = call_api("/health")
preview(health)

{
  "status": "ok"
}


## 5) Boundary and Street Spatial Search

This chapter tests the geometry-aware variants of the top places and restaurant list endpoints.

- Boundary searches use `ST_Covers` and require Polygon or MultiPolygon GeoJSON.
- Street searches use `ST_DWithin` and require LineString or MultiLineString GeoJSON plus `radius_m`.
- The geometry payload is sent as a JSON-encoded query parameter.

In [4]:
def geometry_params(search_type: str, geometry: dict[str, Any], radius_m: int | None = None) -> dict[str, Any]:
    params: dict[str, Any] = {
        "search_type": search_type,
        "geometry": json.dumps(geometry, separators=(",", ":")),
        "venue_type": "",
        "score_basis": 0,
        "score_tier": 0,
    }
    # Omit list filters when no filter is intended. Sending cuisine="" or cost=""
    # makes FastAPI receive [""] and the backend treats that as a literal value.
    if radius_m is not None:
        params["radius_m"] = radius_m
    return params

boundary_geometry = {
    "type": "Polygon",
    "coordinates": [[
        [-0.15, 51.50],
        [-0.10, 51.50],
        [-0.10, 51.52],
        [-0.15, 51.52],
        [-0.15, 51.50],
    ]],
}

street_geometry = {
    "type": "LineString",
    "coordinates": [
        [-0.142, 51.501],
        [-0.130, 51.505],
        [-0.118, 51.510],
    ],
}


def call_api_expect_status(path: str, params: dict[str, Any], expected_status: int) -> requests.Response:
    response = requests.get(f"{API_BASE}{path}", params=params, timeout=TIMEOUT_SECONDS)
    if response.status_code != expected_status:
        raise AssertionError(
            f"Expected HTTP {expected_status}, got {response.status_code}: {response.text}"
        )
    return response


boundary_params = geometry_params("boundary", boundary_geometry)
boundary_top_places = call_api("/api/places/top", params={**boundary_params, "limit": 10})
boundary_restaurants = call_api(
    "/api/places/list",
    params={**boundary_params, "page": 1, "page_size": 10},
)

print("Boundary top places:")
preview(boundary_top_places)
print("\nBoundary restaurant list:")
preview(boundary_restaurants)

street_params = geometry_params("street", street_geometry, radius_m=250)
street_top_places = call_api("/api/places/top", params={**street_params, "limit": 10})
street_restaurants = call_api(
    "/api/places/list",
    params={**street_params, "page": 1, "page_size": 10},
)

print("Street top places:")
preview(street_top_places)
print("\nStreet restaurant list:")
preview(street_restaurants)

Boundary top places:
Summary:
{
  "total": 10,
  "limit": 10
}

Data preview:
[
  {
    "id": "ChIJ5x5n-d0bdkgR5wLl0cOlPEM",
    "restaurant_name": "Tofu Vegan Charlotte Street",
    "cuisine_type": "Vegetarian & Vegan",
    "lat": 51.5190574,
    "lon": -0.1350817,
    "normal_1": 0.9999451637268066,
    "rank": 4
  },
  {
    "id": "ChIJMZ-GUpAbdkgRFXrN8dsAPUM",
    "restaurant_name": "Sale e Pepe Mare at The Langham, London",
    "cuisine_type": "Italian",
    "lat": 51.5175898,
    "lon": -0.143232,
    "normal_1": 0.9979438781738281,
    "rank": 4
  },
  {
    "id": "ChIJ6eVCtnobdkgRyJ0pDxEnD68",
    "restaurant_name": "Tikkas Oxford Circus",
    "cuisine_type": "South Asian",
    "lat": 51.5165939,
    "lon": -0.1398125999999999,
    "normal_1": 0.9958876967430115,
    "rank": 4
  }
]

Data length: 10

Boundary restaurant list:
Summary:
{
  "page": 1,
  "page_size": 10
}

Data preview:
[
  {
    "id": "ChIJ5x5n-d0bdkgR5wLl0cOlPEM",
    "lat": 51.5190574,
    "lon": -0.1350817,
  

In [5]:
invalid_geometry_params = geometry_params(
    "boundary",
    {"type": "LineString", "coordinates": [[-0.13, 51.50], [-0.12, 51.51]]},
)
call_api_expect_status("/api/places/top", invalid_geometry_params, 422)

missing_radius_params = geometry_params("street", street_geometry)
call_api_expect_status("/api/places/list", missing_radius_params, 422)

print("Validation checks passed: unsupported geometry and missing street radius return HTTP 422.")

Validation checks passed: unsupported geometry and missing street radius return HTTP 422.
